# Imports and Paths

In [2]:
# ============================================================
# ADVANCED ANALYTICS
# Mutual Fund Analytics Project
# ============================================================

from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

print("Libraries loaded successfully.")

DATA_PATH = Path("../Data/processed")

print("Current notebook directory:", Path.cwd())
print("Data path:", DATA_PATH.resolve())

print("\nFiles available:")

if DATA_PATH.exists():
    for file in sorted(DATA_PATH.glob("*.csv")):
        print(" -", file.name)
else:
    print("ERROR: Data path does not exist.")

Libraries loaded successfully.
Current notebook directory: c:\Users\vansh\OneDrive\Desktop\Mutual Fund Analytics\notebooks
Data path: C:\Users\vansh\OneDrive\Desktop\Mutual Fund Analytics\Data\Processed

Files available:
 - 01_fund_metadata.csv
 - 02_nav_history.csv
 - 03_aum_by_fund_house.csv
 - 04_monthly_sip_inflows.csv
 - 05_category_inflows.csv
 - 06_industry_folio_count.csv
 - 07_scheme_performance.csv
 - 08_investor_transactions.csv
 - 09_portfolio_holdings.csv
 - 10_benchmark_indices.csv


## Loading Datasets

In [3]:
from pathlib import Path
import pandas as pd

# ============================================================
# DATA PATH
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "Data" / "processed"

print("Data path:")
print(DATA_PATH)

print("\nFiles available:")
for file in sorted(DATA_PATH.glob("*.csv")):
    print(file.name)

Data path:
c:\Users\vansh\OneDrive\Desktop\Mutual Fund Analytics\Data\processed

Files available:
01_fund_metadata.csv
02_nav_history.csv
03_aum_by_fund_house.csv
04_monthly_sip_inflows.csv
05_category_inflows.csv
06_industry_folio_count.csv
07_scheme_performance.csv
08_investor_transactions.csv
09_portfolio_holdings.csv
10_benchmark_indices.csv


In [4]:
# ============================================================
# LOAD DATASETS
# ============================================================

fund_metadata = pd.read_csv(
    DATA_PATH / "01_fund_metadata.csv"
)

nav_history = pd.read_csv(
    DATA_PATH / "02_nav_history.csv"
)

aum = pd.read_csv(
    DATA_PATH / "03_aum_by_fund_house.csv"
)

sip = pd.read_csv(
    DATA_PATH / "04_monthly_sip_inflows.csv"
)

category = pd.read_csv(
    DATA_PATH / "05_category_inflows.csv"
)

folio = pd.read_csv(
    DATA_PATH / "06_industry_folio_count.csv"
)

scheme = pd.read_csv(
    DATA_PATH / "07_scheme_performance.csv"
)

investor = pd.read_csv(
    DATA_PATH / "08_investor_transactions.csv"
)

portfolio = pd.read_csv(
    DATA_PATH / "09_portfolio_holdings.csv"
)

benchmark = pd.read_csv(
    DATA_PATH / "10_benchmark_indices.csv"
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [5]:
#validation cell.
datasets = {
    "fund_metadata": fund_metadata,
    "nav_history": nav_history,
    "aum": aum,
    "sip": sip,
    "category": category,
    "folio": folio,
    "scheme": scheme,
    "investor": investor,
    "portfolio": portfolio,
    "benchmark": benchmark
}

for name, df in datasets.items():
    print(f"{name:20} {df.shape}")

fund_metadata        (35, 7)
nav_history          (97828, 3)
aum                  (90, 5)
sip                  (48, 6)
category             (144, 3)
folio                (21, 6)
scheme               (40, 19)
investor             (32778, 13)
portfolio            (322, 8)
benchmark            (8050, 3)


In [6]:
# Basic Standardization
nav_history.columns = (
    nav_history.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

fund_metadata.columns = (
    fund_metadata.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

investor.columns = (
    investor.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

portfolio.columns = (
    portfolio.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

scheme.columns = (
    scheme.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Columns standardized.")

Columns standardized.


## Preparing NAV returns

In [7]:
# DAILY RETURNS
nav_history["date"] = pd.to_datetime(
    nav_history["date"],
    errors="coerce"
)

nav_history["nav"] = pd.to_numeric(
    nav_history["nav"],
    errors="coerce"
)

nav_history["scheme_code"] = pd.to_numeric(
    nav_history["scheme_code"],
    errors="coerce"
)

nav_history = nav_history.dropna(
    subset=[
        "date",
        "nav",
        "scheme_code"
    ]
)

nav_history = nav_history[
    nav_history["nav"] > 0
].copy()

nav_history = nav_history.sort_values(
    ["scheme_code", "date"]
)

nav_history["daily_return"] = (
    nav_history
    .groupby("scheme_code")["nav"]
    .pct_change()
)

returns = nav_history.dropna(
    subset=["daily_return"]
).copy()

print(
    "Valid schemes:",
    returns["scheme_code"].nunique()
)

print(
    "Return observations:",
    len(returns)
)

Valid schemes: 34
Return observations: 97794


## Historical VaR/CVaR

In [8]:
var_results = []

for scheme_code, group in returns.groupby("scheme_code"):

    daily_returns = group["daily_return"].dropna()

    var_95 = daily_returns.quantile(0.05)

    tail_returns = daily_returns[
        daily_returns <= var_95
    ]

    cvar_95 = tail_returns.mean()

    scheme_name = (
        group["scheme_name"].iloc[0]
        if "scheme_name" in group.columns
        else str(scheme_code)
    )

    var_results.append({
        "scheme_code": scheme_code,
        "scheme_name": scheme_name,
        "observations": len(daily_returns),
        "VaR_95": var_95,
        "CVaR_95": cvar_95
    })

var_cvar = pd.DataFrame(var_results)

var_cvar = var_cvar.sort_values(
    "VaR_95"
).reset_index(drop=True)

display(var_cvar)

,scheme_code,scheme_name,observations,VaR_95,CVaR_95
0,100033,100033,5005,-0.0201,-0.0321
1,119599,119599,872,-0.0185,-0.0260
2,120843,120843,3345,-0.0178,-0.0270
3,120842,120842,3345,-0.0178,-0.0270
4,149323,149323,1145,-0.0166,-0.0246
5,149324,149324,1145,-0.0166,-0.0239
6,149322,149322,1145,-0.0166,-0.0244
7,118634,118634,3341,-0.0164,-0.0274
8,119095,119095,3344,-0.0159,-0.0262
9,118633,118633,3341,-0.0159,-0.0246


In [9]:
# ATTACH FUND METADATA
metadata_lookup = fund_metadata[
    [
        "scheme_code",
        "scheme_name",
        "fund_house"
    ]
].copy()

metadata_lookup["scheme_code"] = pd.to_numeric(
    metadata_lookup["scheme_code"],
    errors="coerce"
)

metadata_lookup = metadata_lookup.drop_duplicates(
    subset=["scheme_code"]
)

if "scheme_name" not in var_cvar.columns:
    
    var_cvar = var_cvar.merge(
        metadata_lookup,
        on="scheme_code",
        how="left",
        validate="one_to_one"
    )

else:
    
    var_cvar = var_cvar.merge(
        metadata_lookup[
            [
                "scheme_code",
                "fund_house"
            ]
        ],
        on="scheme_code",
        how="left",
        validate="one_to_one"
    )

display(var_cvar)

,scheme_code,scheme_name,observations,VaR_95,CVaR_95,fund_house
0,100033,100033,5005,-0.0201,-0.0321,Aditya Birla Sun Life Mutual Fund
1,119599,119599,872,-0.0185,-0.0260,Sundaram Mutual Fund
2,120843,120843,3345,-0.0178,-0.0270,quant Mutual Fund
3,120842,120842,3345,-0.0178,-0.0270,quant Mutual Fund
4,149323,149323,1145,-0.0166,-0.0246,ITI Mutual Fund
5,149324,149324,1145,-0.0166,-0.0239,ITI Mutual Fund
6,149322,149322,1145,-0.0166,-0.0244,ITI Mutual Fund
7,118634,118634,3341,-0.0164,-0.0274,Nippon India Mutual Fund
8,119095,119095,3344,-0.0159,-0.0262,DSP Mutual Fund
9,118633,118633,3341,-0.0159,-0.0246,Nippon India Mutual Fund


In [10]:
# EXPORT VaR / CVaR REPORT
REPORT_PATH = Path("../reports")

REPORT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

var_cvar.to_csv(
    REPORT_PATH / "var_cvar_report.csv",
    index=False
)

print(
    "Saved:",
    REPORT_PATH / "var_cvar_report.csv"
)

Saved: ..\reports\var_cvar_report.csv


## Rolling Sharpe

In [11]:
# ROLLING 90-DAY SHARPE
rolling_data = returns.copy()

rolling_data = rolling_data.sort_values(
    ["scheme_code", "date"]
)

rolling_data["rolling_mean_90"] = (
    rolling_data
    .groupby("scheme_code")["daily_return"]
    .transform(
        lambda x: x.rolling(90).mean()
    )
)

rolling_data["rolling_std_90"] = (
    rolling_data
    .groupby("scheme_code")["daily_return"]
    .transform(
        lambda x: x.rolling(90).std()
    )
)

rolling_data["rolling_sharpe_90"] = (
    rolling_data["rolling_mean_90"]
    / rolling_data["rolling_std_90"]
) * np.sqrt(252)

print(
    "Rolling Sharpe calculated."
)

print(
    "Valid rolling observations:",
    rolling_data["rolling_sharpe_90"].notna().sum()
)

Rolling Sharpe calculated.
Valid rolling observations: 93338


In [12]:
# VERIFY ROLLING SHARPE
print(
    "Schemes:",
    rolling_data["scheme_code"].nunique()
)

display(
    rolling_data[
        [
            "date",
            "scheme_code",
            "daily_return",
            "rolling_sharpe_90"
        ]
    ]
    .dropna(subset=["rolling_sharpe_90"])
    .head(20)
)

Schemes: 34


,date,scheme_code,daily_return,rolling_sharpe_90
90,2006-08-11,100033,0.0044,-0.9656
91,2006-08-14,100033,0.0092,-0.9142
92,2006-08-16,100033,0.0115,-0.8630
93,2006-08-17,100033,-0.0001,-0.7567
94,2006-08-18,100033,0.0034,-0.7812
95,2006-08-21,100033,0.0053,-0.5109
96,2006-08-22,100033,-0.0043,-0.3748
97,2006-08-23,100033,-0.0066,-0.5763
98,2006-08-24,100033,0.0084,-0.6522
99,2006-08-25,100033,0.0043,-0.6540


## Cohort Analysis

In [13]:
investor.columns.tolist()

['investor_id',
 'transaction_date',
 'amfi_code',
 'transaction_type',
 'amount_inr',
 'state',
 'city',
 'city_tier',
 'age_group',
 'gender',
 'annual_income_lakh',
 'payment_mode',
 'kyc_status']

In [14]:
# Preparing investor cohort data
investor_cohort = investor.copy()

# Ensure transaction date is datetime
investor_cohort["transaction_date"] = pd.to_datetime(
    investor_cohort["transaction_date"],
    errors="coerce"
)

# Ensure amount is numeric
investor_cohort["amount_inr"] = pd.to_numeric(
    investor_cohort["amount_inr"],
    errors="coerce"
)

# Remove invalid records
investor_cohort = investor_cohort.dropna(
    subset=["investor_id", "transaction_date", "amount_inr"]
)

# Sort transactions chronologically
investor_cohort = investor_cohort.sort_values(
    ["investor_id", "transaction_date"]
)

print("Valid investor transactions:", len(investor_cohort))
print(
    "Date range:",
    investor_cohort["transaction_date"].min(),
    "to",
    investor_cohort["transaction_date"].max()
)

Valid investor transactions: 13048
Date range: 2024-01-01 00:00:00 to 2025-12-05 00:00:00


In [15]:
# Identify each investor's cohort
first_transactions = (
    investor_cohort
    .groupby("investor_id", as_index=False)["transaction_date"]
    .min()
    .rename(columns={"transaction_date": "first_transaction_date"})
)

first_transactions["cohort_year"] = (
    first_transactions["first_transaction_date"]
    .dt.year
)

# Add cohort information back to every transaction
investor_cohort = investor_cohort.merge(
    first_transactions[
        ["investor_id", "first_transaction_date", "cohort_year"]
    ],
    on="investor_id",
    how="left",
    validate="many_to_one"
)

print(
    "Cohort years:",
    sorted(investor_cohort["cohort_year"].dropna().unique())
)

investor_cohort.head()

Cohort years: [np.int32(2024), np.int32(2025)]


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status,first_transaction_date,cohort_year
0,INV000001,2024-04-11,120505,SIP,44856,Haryana,Gurugram,T30,36-45,Male,19.9000,UPI,Verified,2024-04-11,2024
1,INV000002,2024-03-10,101207,Lumpsum,203094,Maharashtra,Pune,T30,46-55,Male,24.0000,Cheque,Verified,2024-03-10,2024
2,INV000003,2025-11-03,149322,SIP,15185,Gujarat,Ahmedabad,T30,26-35,Female,10.6000,Mandate,Verified,2025-11-03,2025
3,INV000004,2024-01-05,148569,Redemption,59469,Punjab,Chandigarh,T30,26-35,Male,20.0000,Cheque,Verified,2024-01-05,2024
4,INV000004,2024-07-07,148569,SIP,9761,Punjab,Chandigarh,T30,26-35,Male,20.0000,UPI,Verified,2024-01-05,2024


In [16]:
# Cohort investment summary
cohort_summary = (
    investor_cohort
    .groupby("cohort_year")
    .agg(
        investors=("investor_id", "nunique"),
        avg_sip_amount=("amount_inr", "mean"),
        total_invested=("amount_inr", "sum")
    )
    .reset_index()
)

cohort_summary

,cohort_year,investors,avg_sip_amount,total_invested
0,2024,3976,"107,823.1520",1348867631
1,2025,416,"97,670.1840",52546559


In [17]:
# Find the top fund preferred by each cohort
cohort_fund_counts = (
    investor_cohort
    .groupby(["cohort_year", "amfi_code"])
    .size()
    .reset_index(name="transaction_count")
)

top_cohort_funds = (
    cohort_fund_counts
    .sort_values(
        ["cohort_year", "transaction_count"],
        ascending=[True, False]
    )
    .drop_duplicates("cohort_year")
)

top_cohort_funds

,cohort_year,amfi_code,transaction_count
22,2024,119599,358
78,2025,149323,24


In [18]:
# Combine everything
cohort_analysis = cohort_summary.merge(
    top_cohort_funds[
        ["cohort_year", "amfi_code", "transaction_count"]
    ],
    on="cohort_year",
    how="left"
)

cohort_analysis

,cohort_year,investors,avg_sip_amount,total_invested,amfi_code,transaction_count
0,2024,3976,"107,823.1520",1348867631,119599,358
1,2025,416,"97,670.1840",52546559,149323,24


In [19]:
# Format the monetary values
cohort_analysis["avg_sip_amount"] = (
    cohort_analysis["avg_sip_amount"].round(2)
)

cohort_analysis["total_invested"] = (
    cohort_analysis["total_invested"].round(2)
)

cohort_analysis

,cohort_year,investors,avg_sip_amount,total_invested,amfi_code,transaction_count
0,2024,3976,"107,823.1500",1348867631,119599,358
1,2025,416,"97,670.1800",52546559,149323,24


In [21]:
# =========================================================
# CREATE INVESTOR COHORT YEAR
# =========================================================

investor["transaction_date"] = pd.to_datetime(
    investor["transaction_date"],
    errors="coerce"
)

# Remove rows where transaction date is invalid
investor = investor.dropna(
    subset=["transaction_date"]
).copy()

# First transaction year for each investor
first_transaction = (
    investor
    .groupby("investor_id")["transaction_date"]
    .min()
    .rename("first_transaction_date")
)

# Attach first transaction date
investor = investor.merge(
    first_transaction,
    on="investor_id",
    how="left"
)

# Extract cohort year
investor["cohort_year"] = (
    investor["first_transaction_date"]
    .dt.year
)

print(investor[
    ["investor_id", "transaction_date",
     "first_transaction_date", "cohort_year"]
].head())

print("\nCohort years:")
print(
    investor["cohort_year"]
    .value_counts()
    .sort_index()
)

  investor_id transaction_date first_transaction_date  cohort_year
0   INV003054       2024-01-01             2024-01-01         2024
1   INV002952       2024-01-01             2024-01-01         2024
2   INV003420       2024-01-01             2024-01-01         2024
3   INV003436       2024-01-01             2024-01-01         2024
4   INV004691       2024-01-01             2024-01-01         2024

Cohort years:
cohort_year
2024    12510
2025      538
Name: count, dtype: int64


In [22]:
# =========================================================
# COHORT → FUND PREFERENCE
# =========================================================

cohort_fund = (
    investor
    .groupby(["cohort_year", "amfi_code"])
    .agg(
        transactions=("investor_id", "count"),
        total_invested=("amount_inr", "sum")
    )
    .reset_index()
)

# Find the most preferred fund in each cohort
top_fund_by_cohort = (
    cohort_fund
    .sort_values(
        ["cohort_year", "transactions"],
        ascending=[True, False]
    )
    .groupby("cohort_year")
    .first()
    .reset_index()
)

top_fund_by_cohort

,cohort_year,amfi_code,transactions,total_invested
0,2024,119599,358,39187173
1,2025,149323,24,1521078


In [27]:
# =========================================================
# ADD FUND NAMES TO COHORT PREFERENCE TABLE
# =========================================================

# Metadata lookup
metadata_lookup = (
    fund_metadata[
        ["scheme_code", "scheme_name"]
    ]
    .drop_duplicates("scheme_code")
)

# Make sure both identifiers have the same numeric type
top_fund_by_cohort["amfi_code"] = pd.to_numeric(
    top_fund_by_cohort["amfi_code"],
    errors="coerce"
)

metadata_lookup["scheme_code"] = pd.to_numeric(
    metadata_lookup["scheme_code"],
    errors="coerce"
)

# Merge using DIFFERENT column names
top_fund_by_cohort = top_fund_by_cohort.merge(
    metadata_lookup,
    left_on="amfi_code",
    right_on="scheme_code",
    how="left"
)

# Display useful columns
top_fund_by_cohort[
    [
        "cohort_year",
        "amfi_code",
        "scheme_name",
        "transactions",
        "total_invested"
    ]
]

,cohort_year,amfi_code,scheme_name,transactions,total_invested
0,2024,119599,Sundaram Entertainment Opportunities Fund -Dir...,358,39187173
1,2025,149323,ITI Banking and Financial Services Fund - Regu...,24,1521078


In [28]:
# Final cohort table
final_cohort_analysis = cohort_analysis.merge(
    top_fund_by_cohort[
        [
            "cohort_year",
            "scheme_name"
        ]
    ],
    on="cohort_year",
    how="left"
)

final_cohort_analysis

,cohort_year,investors,avg_sip_amount,total_invested,amfi_code,transaction_count,scheme_name
0,2024,3976,"107,823.1500",1348867631,119599,358,Sundaram Entertainment Opportunities Fund -Dir...
1,2025,416,"97,670.1800",52546559,149323,24,ITI Banking and Financial Services Fund - Regu...


# SIP Continuity Analysis

## Prepare SIP Transactions

In [29]:
sip_investor = investor.copy()

# Convert date
sip_investor["transaction_date"] = pd.to_datetime(
    sip_investor["transaction_date"],
    errors="coerce"
)

# Remove invalid dates
sip_investor = sip_investor.dropna(
    subset=["transaction_date"]
)

# Keep only SIP transactions
sip_investor = sip_investor[
    sip_investor["transaction_type"]
    .astype(str)
    .str.upper()
    .str.contains("SIP", na=False)
]

# Sort chronologically for each investor
sip_investor = sip_investor.sort_values(
    ["investor_id", "transaction_date"]
)

print("SIP transactions:", len(sip_investor))
print("Unique SIP investors:", sip_investor["investor_id"].nunique())

SIP transactions: 7824
Unique SIP investors: 3753


In [30]:
# Calculate gap between consecutive SIP transactions
sip_investor["gap_days"] = (
    sip_investor
    .groupby("investor_id")["transaction_date"]
    .diff()
    .dt.days
)

# Calculate SIP transaction statistics per investor
sip_continuity = (
    sip_investor
    .groupby("investor_id")
    .agg(
        sip_transactions=("transaction_date", "count"),
        avg_gap_days=("gap_days", "mean"),
        first_sip_date=("transaction_date", "min"),
        last_sip_date=("transaction_date", "max")
    )
    .reset_index()
)

print(sip_continuity.head())
print("Total investors:", len(sip_continuity))

  investor_id  sip_transactions  avg_gap_days first_sip_date last_sip_date
0   INV000001                 1           NaN     2024-04-11    2024-04-11
1   INV000003                 1           NaN     2025-11-03    2025-11-03
2   INV000004                 3       60.0000     2024-07-07    2024-11-04
3   INV000005                 2      274.0000     2024-02-10    2024-11-10
4   INV000006                 4      214.3333     2024-02-01    2025-11-05
Total investors: 3753


In [31]:
# Keep investors with at least 6 SIP transactions
sip_continuity_6plus = sip_continuity[
    sip_continuity["sip_transactions"] >= 6
].copy()

print(
    "Investors with 6+ SIP transactions:",
    len(sip_continuity_6plus)
)

Investors with 6+ SIP transactions: 38


In [32]:
# Flag investors with average SIP gap > 35 days
sip_continuity_6plus["continuity_status"] = np.where(
    sip_continuity_6plus["avg_gap_days"] > 35,
    "At-Risk",
    "Consistent"
)

sip_continuity_6plus.head(10)

,investor_id,sip_transactions,avg_gap_days,first_sip_date,last_sip_date,continuity_status
17,INV000023,6,73.2000,2024-05-04,2025-05-05,At-Risk
66,INV000086,6,49.2000,2024-05-02,2025-01-03,At-Risk
83,INV000105,6,115.6000,2024-02-01,2025-09-01,At-Risk
87,INV000109,6,83.6000,2024-03-11,2025-05-03,At-Risk
114,INV000148,6,61.4000,2024-03-04,2025-01-05,At-Risk
174,INV000228,7,45.6667,2024-05-07,2025-02-05,At-Risk
214,INV000279,7,111.5000,2024-01-03,2025-11-02,At-Risk
345,INV000452,6,108.6000,2024-03-09,2025-09-03,At-Risk
519,INV000676,8,81.4286,2024-05-10,2025-12-01,At-Risk
766,INV001002,7,51.3333,2024-01-04,2024-11-07,At-Risk


In [33]:
# key statistics
total_eligible = len(sip_continuity_6plus)

at_risk = (
    sip_continuity_6plus["continuity_status"]
    .eq("At-Risk")
    .sum()
)

consistent = (
    sip_continuity_6plus["continuity_status"]
    .eq("Consistent")
    .sum()
)

at_risk_rate = (
    at_risk / total_eligible * 100
    if total_eligible > 0 else 0
)

print("==========================================")
print("SIP CONTINUITY ANALYSIS")
print("==========================================")
print("Eligible investors:", total_eligible)
print("Consistent investors:", consistent)
print("At-Risk investors:", at_risk)
print(f"At-Risk rate: {at_risk_rate:.2f}%")

SIP CONTINUITY ANALYSIS
Eligible investors: 38
Consistent investors: 0
At-Risk investors: 38
At-Risk rate: 100.00%


In [34]:
# At risk investors
at_risk_investors = (
    sip_continuity_6plus[
        sip_continuity_6plus["continuity_status"] == "At-Risk"
    ]
    .sort_values("avg_gap_days", ascending=False)
)

at_risk_investors.head(20)

,investor_id,sip_transactions,avg_gap_days,first_sip_date,last_sip_date,continuity_status
2382,INV003187,6,140.6000,2024-01-02,2025-12-05,At-Risk
912,INV001202,6,138.2000,2024-01-11,2025-12-02,At-Risk
2765,INV003670,6,134.8000,2024-01-01,2025-11-05,At-Risk
1502,INV002019,6,133.6000,2024-02-06,2025-12-05,At-Risk
2767,INV003673,6,127.0000,2024-03-10,2025-12-05,At-Risk
1648,INV002219,6,125.6000,2024-02-12,2025-11-01,At-Risk
3108,INV004125,6,122.2000,2024-02-02,2025-10-05,At-Risk
3046,INV004038,6,122.0000,2024-03-05,2025-11-05,At-Risk
882,INV001160,6,121.8000,2024-02-04,2025-10-05,At-Risk
2036,INV002733,6,120.2000,2024-02-12,2025-10-05,At-Risk


In [35]:
# Saving Anlysis
sip_continuity_6plus.to_csv(
    "../reports/sip_continuity_analysis.csv",
    index=False
)

print("Saved: sip_continuity_analysis.csv")

Saved: sip_continuity_analysis.csv
